### Proyecto final NLP

### 
LIBRERIAS

In [16]:
import pandas as pd
import numpy as np
import re
import nltk
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# NLP clásico
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Deep Learning
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

# Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

####
DOWNLOADS

In [2]:
# Descargar recursos NLTK
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Víctor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Víctor\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

#### CARGA DE DATOS

In [3]:
df = pd.read_csv("yelp.csv")

print(df.head())
print(df.info())

              business_id        date               review_id  stars  \
0  9yKzy9PApeiPPOUJEtnvkg  2011-01-26  fWKvX83p0-ka4JS3dc6E5A      5   
1  ZRJwVLyzEJq1VAihDhYiow  2011-07-27  IjZ33sJrzXqU-0X6U8NwyA      5   
2  6oRAC4uyJCsJl1X0WZpVSA  2012-06-14  IESLBzqUCLdSzSqm0eCSxQ      4   
3  _1QQZuf4zZOyFCvXc0o6Vg  2010-05-27  G-WvGaISbqqaMHlNnByodA      5   
4  6ozycU1RpktNG2-1BroVtw  2012-01-05  1uJFq2r5QfJG_6ExMRCaGw      5   

                                                text    type  \
0  My wife took me here on my birthday for breakf...  review   
1  I have no idea why some people give bad review...  review   
2  love the gyro plate. Rice is so good and I als...  review   
3  Rosie, Dakota, and I LOVE Chaparral Dog Park!!...  review   
4  General Manager Scott Petello is a good egg!!!...  review   

                  user_id  cool  useful  funny  
0  rLtl8ZkDX5vH5nAx9C3q5Q     2       5      0  
1  0a2KyEL0d3Yb1V6aivbIuQ     0       0      0  
2  0hT2KtfLiobPvh6cDC8JQg     0    

#### CREACION DE VARIABLE OBJETIVO

In [4]:
def map_sentiment(stars):
    if stars <= 2:
        return 0  # negativo
    elif stars == 3:
        return 1  # neutral
    else:
        return 2  # positivo

df['sentiment'] = df['stars'].apply(map_sentiment)

print(df['sentiment'].value_counts())

sentiment
2    6863
0    1676
1    1461
Name: count, dtype: int64


#### 
PREPROCESAMIENTO

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

df['clean_text'] = df['text'].apply(clean_text)

### 
SPLIT

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)


###
TOKENIZACION Y SECUENCAIS

In [7]:
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

### 
MODELO LSTM

In [ ]:
model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# Entrenamiento
model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

# Evaluación LSTM
y_pred_lstm = np.argmax(model.predict(X_test_pad), axis=1)

print("\n LSTM RESULTS ")
print(classification_report(y_test, y_pred_lstm))
print(confusion_matrix(y_test, y_pred_lstm))


c:\Users\Víctor\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
225/225 ━━━━━━━━━━━━━━━━━━━━ 12s 44ms/step - accuracy: 0.7149 - loss: 0.7290 - val_accuracy: 0.7713 - val_loss: 0.6072
Epoch 2/5
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.8082 - loss: 0.4841 - val_accuracy: 0.7675 - val_loss: 0.5865
Epoch 3/5
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.8819 - loss: 0.3070 - val_accuracy: 0.7375 - val_loss: 0.6837
Epoch 4/5
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.9456 - loss: 0.1632 - val_accuracy: 0.7437 - val_loss: 0.9552
Epoch 5/5
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.9671 - loss: 0.0931 - val_accuracy: 0.7325 - val_loss: 1.1441
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step

=== LSTM RESULTS ===
              precision    recall  f1-score   support

           0       0.69      0.49      0.57       335
           1       0.34      0.35      0.34       292
           2       0.84      0.89      0.86      1373

    accuracy                           0.74      2000
   macro avg       0.62 

In [10]:
import transformers
print(transformers.__version__)

5.3.0


###
TRANSFORMER

In [11]:
model_name = "distilbert-base-uncased"

tokenizer_bert = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(texts):
    return tokenizer_bert(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_function(X_train)
test_encodings = tokenize_function(X_test)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.values

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, y_train)
test_dataset = Dataset(test_encodings, y_test)

model_bert = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    eval_strategy="epoch"
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

preds = trainer.predict(test_dataset)
y_pred_bert = np.argmax(preds.predictions, axis=1)

print("\nBERT RESULTS")
print(classification_report(y_test, y_pred_bert))
print(confusion_matrix(y_test, y_pred_bert))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9594.00it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\Víctor\AppData\Local\Programs\Python\Python313\Lib\si

Epoch,Training Loss,Validation Loss
1,0.592406,0.550858
2,0.428799,0.571538


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.47it/s]
c:\Users\Víctor\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.27it/s]
c:\Users\Víctor\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



=== BERT RESULTS ===
              precision    recall  f1-score   support

           0       0.77      0.67      0.71       335
           1       0.40      0.33      0.36       292
           2       0.86      0.92      0.89      1373

    accuracy                           0.79      2000
   macro avg       0.68      0.64      0.65      2000
weighted avg       0.78      0.79      0.78      2000

[[ 223   58   54]
 [  46   97  149]
 [  22   88 1263]]


####
COMPARACIÓN FINAL

In [ ]:
# MÉTRICAS LSTM
acc_lstm = accuracy_score(y_test, y_pred_lstm)
prec_lstm = precision_score(y_test, y_pred_lstm, average="weighted")
rec_lstm = recall_score(y_test, y_pred_lstm, average="weighted")
f1_lstm = f1_score(y_test, y_pred_lstm, average="weighted")

print(" LSTM ")
print("Accuracy:", acc_lstm)
print("Precision:", prec_lstm)
print("Recall:", rec_lstm)
print("F1:", f1_lstm)
print(classification_report(y_test, y_pred_lstm))
print(confusion_matrix(y_test, y_pred_lstm))

# MÉTRICAS BERT
acc_bert = accuracy_score(y_test, y_pred_bert)
prec_bert = precision_score(y_test, y_pred_bert, average="weighted")
rec_bert = recall_score(y_test, y_pred_bert, average="weighted")
f1_bert = f1_score(y_test, y_pred_bert, average="weighted")

print(" DISTILBERT ")
print("Accuracy:", acc_bert)
print("Precision:", prec_bert)
print("Recall:", rec_bert)
print("F1:", f1_bert)
print(classification_report(y_test, y_pred_bert))
print(confusion_matrix(y_test, y_pred_bert))

=== LSTM ===
Accuracy: 0.7435
Precision: 0.7390450626788969
Recall: 0.7435
F1: 0.7376929685479598
              precision    recall  f1-score   support

           0       0.69      0.49      0.57       335
           1       0.34      0.35      0.34       292
           2       0.84      0.89      0.86      1373

    accuracy                           0.74      2000
   macro avg       0.62      0.58      0.59      2000
weighted avg       0.74      0.74      0.74      2000

[[ 164   85   86]
 [  35  102  155]
 [  37  115 1221]]
=== DISTILBERT ===
Accuracy: 0.7915
Precision: 0.7780778914426665
Recall: 0.7915
F1: 0.7830924313250176
              precision    recall  f1-score   support

           0       0.77      0.67      0.71       335
           1       0.40      0.33      0.36       292
           2       0.86      0.92      0.89      1373

    accuracy                           0.79      2000
   macro avg       0.68      0.64      0.65      2000
weighted avg       0.78      0.79   

In [15]:
comparacion = pd.DataFrame({
    "Modelo": ["LSTM", "DistilBERT"],
    "Accuracy": [acc_lstm, acc_bert],
    "Precision": [prec_lstm, prec_bert],
    "Recall": [rec_lstm, rec_bert],
    "F1-score": [f1_lstm, f1_bert]
})

print("\n TABLA COMPARATIVA ")
print(comparacion)


 TABLA COMPARATIVA 
       Modelo  Accuracy  Precision  Recall  F1-score
0        LSTM    0.7435   0.739045  0.7435  0.737693
1  DistilBERT    0.7915   0.778078  0.7915  0.783092


In [17]:
print(comparacion.round(4))

       Modelo  Accuracy  Precision  Recall  F1-score
0        LSTM    0.7435     0.7390  0.7435    0.7377
1  DistilBERT    0.7915     0.7781  0.7915    0.7831


# Conclusiones

##  Comparación de resultados


###
En base a los resultados obtenidos, se concluye que el modelo DistilBERT es superior al modelo LSTM en la tarea de clasificación de sentimiento, logrando mejores resultados en todas las métricas evaluadas.

No obstante, el modelo LSTM sigue siendo una alternativa válida en escenarios donde los recursos computacionales son limitados o se requiere una solución más ligera.

En definitiva, el uso de modelos basados en Transformers representa actualmente el enfoque más eficaz para tareas de procesamiento del lenguaje natural que requieren una comprensión profunda del contexto.

## Comparación de modelos
####
El modelo LSTM y el modelo basado en Transformer presentan diferencias significativas en su rendimiento y comportamiento.

El modelo LSTM muestra un rendimiento aceptable en términos de accuracy, pero presenta limitaciones en la clasificación de clases minoritarias, especialmente la clase neutral. Esto se debe principalmente al desbalanceo del dataset y a la limitada capacidad del modelo para capturar relaciones semánticas complejas.

Por otro lado, el modelo basado en DistilBERT, al utilizar representaciones contextuales del lenguaje, es capaz de captar mejor el significado del texto, lo que se traduce en un mejor rendimiento global, especialmente en tareas de clasificación semántica.

Sin embargo, este modelo requiere un mayor coste computacional y un proceso de entrenamiento más complejo.

En conclusión, el modelo Transformer ofrece mejores resultados en términos de rendimiento, mientras que el modelo LSTM representa una solución más eficiente desde el punto de vista computacional.

## Limitaciones

El principal problema identificado en este proyecto es el desbalanceo de clases, que afecta negativamente al rendimiento del modelo en las clases minoritarias.

## Líneas futuras

Como mejora futura, se propone:

aplicar técnicas de balanceo de datos
optimizar hiperparámetros
entrenar modelos más avanzados como RoBERTa